# Citadel JWT Authentication & App Role Authorization - Testing Center

## Validate JWT-based Authentication and Role-Based Access Control Across All 3 LLM API Endpoints

Use this Jupyter notebook to verify unified JWT authentication and app role authorization in Citadel Access Contracts, including:
- **Unified Security Handler**: Single `security-handler` fragment across Azure OpenAI, Universal LLM, and Unified AI APIs
- **API Key + JWT (enforced)**: Access contracts that require both API key and valid JWT Bearer token
- **App Role Authorization**: Access contracts that require specific app roles (e.g., `Models.Read`, `MCP.Read`, `Agent.Read`) in the JWT `roles` claim
- **All 3 API Flavors**: Testing Azure OpenAI API, Universal LLM API, and Unified AI API endpoints
- **Negative Testing**: Verifying that requests without required JWT or app roles are rejected
- **Dual Auth Flows**: Client credentials flow (service-to-service) and MSAL device code flow (interactive user sign-in)

> **Note:** This notebook assumes a fully deployed Citadel Governance Hub with JWT authentication configured.
> The gateway validates tokens using APIM named values (`JWT-OpenIdConfigUrl`, `JWT-Issuer`, `JWT-AppRegistrationId`)
> or per-product custom overrides (`jwtAudience`, `jwtIssuer`, `jwtOpenIdConfigUrl`) set in the product policy.
>
> **Setup references:**
> - [JWT Authentication Guide](../guides/entraid-auth-validation.md) - Gateway-side JWT configuration (named values & custom overrides)
> - [JWT Client Identity & Permissions Guide](../guides/jwt-client-identity-permissions.md) - Configuring client identities, group-based access, and role assignment
> - [Entra ID Setup](../bicep/infra/entra-id-setup/) - Auto-provisioning Entra ID app registration with app roles

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

In [ ]:
import os
import sys, json, requests, time, base64
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# CITADEL GOVERNANCE HUB CONFIGURATION
# ============================================================================
governance_hub_resource_group = "REPLACE"  ## specify the resource group name where the Governance Hub is located
location = "REPLACE"  ## specify the location of the Governance Hub

# ============================================================================
# ENTRA ID / OAUTH CONFIGURATION
# These values are used to acquire JWT tokens for testing.
# For Entra ID: auto-populated by entra-id-setup/setup.ps1
# For other providers: set manually with your OAuth provider's values
# ============================================================================
entra_tenant_id = "REPLACE"       # OAuth tenant/domain ID
entra_client_id = "REPLACE"       # OAuth client/application ID
entra_client_secret = "REPLACE"   # OAuth client secret
entra_audience = f"api://{entra_client_id}"        # Expected audience (e.g., api://{client-id})

# Token endpoint (Entra ID default, change for other providers)
token_endpoint = f"https://login.microsoftonline.com/{entra_tenant_id}/oauth2/v2.0/token"

# ============================================================================
# API & MODEL CONFIGURATION
# ============================================================================
model_name = "gpt-4.1"  # Model to use in tests
openai_api_version = "2024-12-01-preview"
inference_api_version = "2024-05-01-preview"

# ============================================================================
# KEY VAULT CONFIGURATION (optional)
# ============================================================================
use_keyvault_integration = False
keyvault_subscription_id = "REPLACE"
keyvault_resource_group = "REPLACE"
keyvault_name = "REPLACE"

<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected with JWT named values configured.

In [ ]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    # Get supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models in APIM policy fragment 'set-backend-pools': {supported_models}")

    # Build endpoint URLs for all 3 API flavors
    # 1. Azure OpenAI API
    apimClientTool.discover_api("openai")
    openai_chat_url = f"{apimClientTool.azure_endpoint}openai/deployments/{model_name}/chat/completions?api-version={openai_api_version}"
    utils.print_info(f"Azure OpenAI Chat URL: {openai_chat_url}")

    # 2. Universal LLM API
    apimClientTool.discover_api("models")
    universal_llm_chat_url = f"{apimClientTool.azure_endpoint}models/chat/completions?api-version={inference_api_version}"
    utils.print_info(f"Universal LLM Chat URL: {universal_llm_chat_url}")

    # 3. Unified AI API
    apimClientTool.discover_api("unified-ai")
    unified_ai_openai_url = f"{apimClientTool.azure_endpoint}unified-ai/openai/deployments/{model_name}/chat/completions?api-version={openai_api_version}"
    unified_ai_models_url = f"{apimClientTool.azure_endpoint}unified-ai/models/chat/completions?api-version={inference_api_version}"
    utils.print_info(f"Unified AI (OpenAI) Chat URL: {unified_ai_openai_url}")
    utils.print_info(f"Unified AI (Models) Chat URL: {unified_ai_models_url}")

    utils.print_ok(f"Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

<a id='3'></a>
### 3️⃣ Acquire JWT Token

Acquire a JWT token from the identity provider using client credentials flow.

In [ ]:
jwt_token = None

token_data = {
    "grant_type": "client_credentials",
    "client_id": entra_client_id,
    "client_secret": entra_client_secret,
    "scope": f"{entra_audience}/.default"
}

utils.print_info(f"Requesting JWT token from: {token_endpoint}")
utils.print_info(f"Client ID: {entra_client_id}")
utils.print_info(f"Audience: {entra_audience}")

try:
    token_response = requests.post(token_endpoint, data=token_data, timeout=30)

    if token_response.status_code == 200:
        token_json = token_response.json()
        jwt_token = token_json["access_token"]
        expires_in = token_json.get("expires_in", "unknown")

        utils.print_ok(f"JWT token acquired successfully!")
        utils.print_info(f"Token length: {len(jwt_token)} characters")
        utils.print_info(f"Expires in: {expires_in} seconds")

        # Decode token payload to inspect claims
        token_parts = jwt_token.split('.')
        if len(token_parts) >= 2:
            payload = token_parts[1] + '=' * (4 - len(token_parts[1]) % 4)
            decoded = json.loads(base64.b64decode(payload))
            utils.print_info(f"Token issuer: {decoded.get('iss', 'N/A')}")
            utils.print_info(f"Token audience: {decoded.get('aud', 'N/A')}")
            utils.print_info(f"Token app ID (azp): {decoded.get('azp', 'N/A')}")

            # Display app roles from the token
            token_roles = decoded.get('roles', [])
            if token_roles:
                utils.print_ok(f"Token roles: {', '.join(token_roles)}")
            else:
                utils.print_info(f"Token roles: none (service principal may not have app roles assigned)")
                utils.print_info(f"  To assign a role: az rest --method POST --uri 'https://graph.microsoft.com/v1.0/servicePrincipals/{{client-sp-id}}/appRoleAssignments' ...")
    else:
        utils.print_error(f"Failed to acquire token: {token_response.status_code}")
        utils.print_error(f"Response: {token_response.text[:500]}")
except Exception as e:
    utils.print_error(f"Error acquiring JWT token: {e}")

<a id='3b'></a>
### 3️⃣.b Acquire JWT Token via Device Code Flow (Interactive)

Use MSAL device code flow for **user sign-in** scenarios. This acquires a delegated token with the `access_as_user` scope.
App roles assigned to the **user** (or their group) will appear in the `roles` claim.

> **Prerequisites:**
> - Install MSAL: `pip install msal`
> - The app registration must have `isFallbackPublicClient` enabled (or use the Entra ID portal to enable "Allow public client flows")
> - The user must be assigned an app role (e.g., `Models.Read`) via Entra ID portal or CLI

In [ ]:
# Device code flow using MSAL (interactive user sign-in)
# Set use_device_flow = True to acquire a user token instead of client credentials
use_device_flow = True
device_flow_token = None

if use_device_flow:
    # Device code flow requires the app to allow public client flows.
    # Enable it if not already set (idempotent PATCH).
    # utils.print_info("Ensuring app registration allows public client flows...")
    # utils.run(
    #     f"az ad app update --id {entra_client_id} --is-fallback-public-client true",
    #     "Public client flows enabled",
    #     "Failed to enable public client flows (you may need to set it manually in Entra ID portal)"
    # )

    try:
        import msal

        app = msal.PublicClientApplication(
            entra_client_id,
            authority=f"https://login.microsoftonline.com/{entra_tenant_id}"
        )

        # Initiate device code flow with the gateway's delegated scope
        flow = app.initiate_device_flow(scopes=[f"{entra_audience}/access_as_user"])
        if "user_code" not in flow:
            utils.print_error(f"Failed to create device flow: {json.dumps(flow, indent=2)}")
        else:
            print(flow["message"])  # Displays: "To sign in, use a web browser to open..."

            result = app.acquire_token_by_device_flow(flow)

            if "access_token" in result:
                device_flow_token = result["access_token"]
                utils.print_ok(f"Device flow token acquired!")
                utils.print_info(f"Token length: {len(device_flow_token)} characters")

                # Decode and display claims including roles
                parts = device_flow_token.split('.')
                if len(parts) >= 2:
                    payload = parts[1] + '=' * (4 - len(parts[1]) % 4)
                    decoded = json.loads(base64.b64decode(payload))
                    utils.print_info(f"User: {decoded.get('preferred_username', decoded.get('upn', 'N/A'))}")
                    utils.print_info(f"Audience: {decoded.get('aud', 'N/A')}")
                    roles = decoded.get('roles', [])
                    if roles:
                        utils.print_ok(f"Roles: {', '.join(roles)}")
                    else:
                        utils.print_info(f"Roles: none (assign roles to the user via Entra ID)")
            else:
                utils.print_error(f"Token acquisition failed: {result.get('error_description', result.get('error', 'unknown'))}")
    except ImportError:
        utils.print_error("MSAL not installed. Run: pip install msal")
    except Exception as e:
        utils.print_error(f"Device flow error: {e}")
else:
    utils.print_info("Device code flow is disabled. Set use_device_flow = True to enable.")

<a id='3c'></a>
### 3️⃣.c Inspect Issued JWT Tokens

Decode and display the full JWT payload for any tokens acquired above. Use [jwt.ms](https://jwt.ms) or [jwt.io](https://jwt.io) for interactive inspection.

In [ ]:
def inspect_jwt(token, label="JWT Token"):
    """Decode and pretty-print a JWT token's header and payload."""
    if not token:
        utils.print_info(f"[{label}] No token to inspect.")
        return

    parts = token.split('.')
    if len(parts) < 2:
        utils.print_error(f"[{label}] Invalid JWT format (expected 3 parts, got {len(parts)})")
        return

    def decode_part(part):
        padded = part + '=' * (4 - len(part) % 4)
        return json.loads(base64.b64decode(padded))

    header = decode_part(parts[0])
    payload = decode_part(parts[1])

    utils.print_info(f"\n{'='*80}")
    utils.print_info(f"🔍 {label}")
    utils.print_info(f"{'='*80}")

    utils.print_info(f"\n--- Header ---")
    print(json.dumps(header, indent=2))

    utils.print_info(f"\n--- Payload ---")
    print(json.dumps(payload, indent=2))

    # Highlight key claims
    utils.print_info(f"\n--- Key Claims ---")
    utils.print_info(f"  aud (audience):  {payload.get('aud', 'N/A')}")
    utils.print_info(f"  iss (issuer):    {payload.get('iss', 'N/A')}")
    utils.print_info(f"  azp (client):    {payload.get('azp', 'N/A')}")
    utils.print_info(f"  sub (subject):   {payload.get('sub', 'N/A')}")
    utils.print_info(f"  upn/email:       {payload.get('preferred_username', payload.get('upn', payload.get('email', 'N/A')))}")

    roles = payload.get('roles', [])
    if roles:
        utils.print_ok(f"  roles:           {', '.join(roles)}")
    else:
        utils.print_info(f"  roles:           none")

    # Token lifetime
    import datetime
    iat = payload.get('iat')
    exp = payload.get('exp')
    if iat and exp:
        issued = datetime.datetime.fromtimestamp(iat, tz=datetime.timezone.utc)
        expires = datetime.datetime.fromtimestamp(exp, tz=datetime.timezone.utc)
        utils.print_info(f"  issued:          {issued.isoformat()}")
        utils.print_info(f"  expires:         {expires.isoformat()}")
        utils.print_info(f"  lifetime:        {(expires - issued).total_seconds() / 60:.0f} minutes")

    # Print raw token for pasting into jwt.ms
    utils.print_info(f"\n--- Raw Token (paste into https://jwt.ms) ---")
    print(token)

# Inspect all acquired tokens
inspect_jwt(jwt_token, "Client Credentials Token (§3)")

if device_flow_token:
    inspect_jwt(device_flow_token, "Device Code Flow Token (§3b)")

<a id='3d'></a>
### 3️⃣.d Select Active Token for Testing

Choose which token to use for all subsequent tests. Change `token_source` to switch between:
- `"client_credentials"` — Service principal token from §3 (service-to-service)
- `"device_flow"` — User token from §3b (interactive sign-in)

In [ ]:
# ============================================================================
# TOKEN SELECTOR — Change this value to switch which token is used in all tests
# ============================================================================
token_source = "client_credentials"  # Options: "client_credentials" or "device_flow"

# Resolve the active token based on selection
if token_source == "device_flow":
    if device_flow_token:
        active_token = device_flow_token
        utils.print_ok(f"Active token: Device Code Flow")
    else:
        utils.print_error("Device flow token not available. Run §3b first or switch to 'client_credentials'.")
        active_token = None
elif token_source == "client_credentials":
    if jwt_token:
        active_token = jwt_token
        utils.print_ok(f"Active token: Client Credentials (§3)")
    else:
        utils.print_error("Client credentials token not available. Run §3 first.")
        active_token = None
else:
    utils.print_error(f"Unknown token_source: '{token_source}'. Use 'client_credentials' or 'device_flow'.")
    active_token = None

# Display summary of the selected token
if active_token:
    parts = active_token.split('.')
    if len(parts) >= 2:
        pl = parts[1] + '=' * (4 - len(parts[1]) % 4)
        claims = json.loads(base64.b64decode(pl))
        roles = claims.get('roles', [])
        utils.print_info(f"  Source:   {token_source}")
        utils.print_info(f"  Audience: {claims.get('aud', 'N/A')}")
        utils.print_info(f"  Identity: {claims.get('azp', claims.get('preferred_username', 'N/A'))}")
        utils.print_info(f"  Roles:    {', '.join(roles) if roles else 'none'}")

---
<a id='use-case-1'></a>
## 🛡️ Use Case 1: JWT-Enforced Access Contract

This use case creates an access contract that requires **both API key and JWT token** for all requests.
The `jwtRequired` variable is set to `"true"` in the product policy, which triggers the `security-handler`
fragment to enforce JWT validation across all 3 API endpoints.

---

<a id='4.1'></a>
### 4️⃣.1 Define JWT Access Contract Configuration

In [ ]:
timestamp = time.strftime('%Y%m%d%H%M%S')

# JWT-Enforced Access Contract Configuration
jwt_contract = {
    "name": f"jwt-auth-contract-{timestamp}",
    "business_unit": "Security",
    "use_case_name": "JwtAuth",
    "environment": "DEV",
    "use_keyvault": use_keyvault_integration,
    "endpoint_secret": "SEC-JWT-LLM-ENDPOINT",
    "apikey_secret": "SEC-JWT-LLM-KEY",
    "description": "Security JWT Auth - Enforced Valid JWT + API Key across all endpoints"
}

jwt_product_id = f"LLM-{jwt_contract['business_unit']}-{jwt_contract['use_case_name']}-{jwt_contract['environment']}"

utils.print_info(f"JWT Access Contract Configuration:")
utils.print_info(f"  Business Unit: {jwt_contract['business_unit']}")
utils.print_info(f"  Use Case: {jwt_contract['use_case_name']}")
utils.print_info(f"  JWT Enforcement: ENABLED")
utils.print_info(f"  Product ID: {jwt_product_id}")

<a id='4.2'></a>
### 4️⃣.2 Create JWT Product Policy

Generate a product policy XML that enables JWT authentication via the unified `security-handler` fragment.

In [ ]:
import shutil

bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

# Create folder structure for JWT contract
contract = jwt_contract
folder_name = f"{contract['business_unit'].lower()}-{contract['use_case_name'].lower()}"
environment_folder = contract['environment'].lower()
jwt_contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(jwt_contract_folder, exist_ok=True)
utils.print_info(f"Created folder: {jwt_contract_folder}")

# Create JWT-enforced Product Policy XML
jwt_product_policy = '''<policies>
    <inbound>
        <base />
        <!-- Enable JWT requirement for this product -->
        <!-- security-handler reads this variable and enforces JWT validation -->
        <set-variable name="jwtRequired" value="true" />

        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Setting allowed models variable (comma-separated list) -->
        <set-variable name="allowedModels" value="gpt-4o,gpt-4o-mini,phi-4" />

        <!-- Validate model access based on allowedModels -->
        <include-fragment fragment-id="validate-model-access" />

        <!-- Capacity management -->
        <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="5000" estimate-prompt-tokens="false" token-quota="100000" token-quota-period="Monthly" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

# Write the policy file
jwt_policy_file = os.path.join(jwt_contract_folder, "ai-product-policy.xml")
with open(jwt_policy_file, 'w') as f:
    f.write(jwt_product_policy)
utils.print_ok(f"JWT product policy file created: {jwt_policy_file}")

<a id='4.3'></a>
### 4️⃣.3 Create Parameter File and Deploy JWT Access Contract

In [ ]:
# Generate parameter file for JWT contract
jwt_params_file = os.path.join(jwt_contract_folder, "main.bicepparam")
policy_relative_path = "ai-product-policy.xml"

jwt_params_content = f'''using '../../../main.bicep'

// ============================================================================
// {contract['description']} - Generated from JWT Authentication Testing Notebook
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '{keyvault_subscription_id}'
  resourceGroupName: '{keyvault_resource_group}'
  name: '{keyvault_name}'
}}

param useTargetAzureKeyVault = {str(contract['use_keyvault']).lower()}

param useCase = {{
  businessUnit: '{contract['business_unit']}'
  useCaseName: '{contract['use_case_name']}'
  environment: '{contract['environment']}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: '{contract['endpoint_secret']}'
    apiKeySecretName: '{contract['apikey_secret']}'
    policyXml: loadTextContent('{policy_relative_path}')
  }}
]

param productTerms = 'JWT Authentication Access Contract - {contract["description"]}'

// Azure AI Foundry Integration (disabled)
param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
'''

with open(jwt_params_file, 'w') as f:
    f.write(jwt_params_content)
utils.print_ok(f"Parameter file created: {jwt_params_file}")

# Deploy the JWT access contract
utils.print_info(f"\n{'='*60}")
utils.print_info(f"Deploying JWT Access Contract...")
utils.print_info(f"{'='*60}")

deployment_cmd = f"az deployment sub create --name {contract['name']} --location {location} --template-file {template_file} --parameters {jwt_params_file}"

jwt_deployment_output = utils.run(
    deployment_cmd,
    f"Deployment '{contract['name']}' succeeded",
    f"Deployment '{contract['name']}' failed"
)

if jwt_deployment_output.success:
    utils.print_ok(f"JWT Access Contract deployed successfully!")
else:
    utils.print_error(f"JWT Access Contract deployment failed!")

<a id='4.4'></a>
### 4️⃣.4 Retrieve API Key for JWT Access Contract

In [ ]:
# Re-initialize APIM client to pick up new subscriptions
apimClientTool.initialize()

jwt_subscription_name = f"{jwt_product_id}-SUB-01"
jwt_api_key = None

for sub in apimClientTool.apim_subscriptions:
    if jwt_subscription_name.lower() in sub.get('name', '').lower():
        jwt_api_key = sub.get('key')
        utils.print_ok(f"Found API key for {jwt_product_id}")
        break

if not jwt_api_key:
    utils.print_error(f"Could not find API key for {jwt_product_id}")

---
## 🧪 Test Suite: JWT Authentication Across All 3 API Endpoints

The following tests validate the unified `security-handler` behavior:
1. **API Key + JWT** on each endpoint (should succeed)
2. **API Key only** on each endpoint (should fail with 401 - JWT required)
3. **JWT only** without API key (should fail with 401 - API key required)
4. **Invalid JWT** with valid API key (should fail with 401 - invalid token)

---

<a id='5.1'></a>
### 5️⃣.1 Define Test Endpoints and Payloads

In [ ]:
# Define the 3 API endpoint configurations
api_endpoints = [
    {
        "name": "Azure OpenAI API",
        "url": openai_chat_url,
        "payload": {
            "messages": [
                {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                {"role": "user", "content": "Say hello in exactly 3 words."}
            ],
            "max_tokens": 20,
            "temperature": 0.3
        }
    },
    {
        "name": "Universal LLM API",
        "url": universal_llm_chat_url,
        "payload": {
            "model": model_name,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                {"role": "user", "content": "Say hello in exactly 3 words."}
            ],
            "max_tokens": 20,
            "temperature": 0.3
        }
    },
    {
        "name": "Unified AI API (OpenAI format)",
        "url": unified_ai_openai_url,
        "payload": {
            "messages": [
                {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                {"role": "user", "content": "Say hello in exactly 3 words."}
            ],
            "max_tokens": 20,
            "temperature": 0.3,
            "model": model_name
        }
    },
    {
        "name": "Unified AI API (Models/Inference format)",
        "url": unified_ai_models_url,
        "payload": {
            "model": model_name,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                {"role": "user", "content": "Say hello in exactly 3 words."}
            ],
            "max_tokens": 20,
            "temperature": 0.3
        }
    }
]

utils.print_info(f"Defined {len(api_endpoints)} API endpoints for testing:")
for ep in api_endpoints:
    utils.print_info(f"  - {ep['name']}: {ep['url'][:80]}...")

<a id='5.2'></a>
### 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)

Send requests to all 3 API endpoints with both API key and valid JWT Bearer token.

In [ ]:
test1_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 1: API Key + JWT Token (Expected: 200 OK)")
utils.print_info(f"{'='*80}")
utils.print_info(f"Using token: {token_source}")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": jwt_api_key,
        "Authorization": f"Bearer {active_token}",
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=60)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code == 200
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            data = response.json()
            content = data.get('choices', [{}])[0].get('message', {}).get('content', 'N/A')
            usage = data.get('usage', {})
            result["response_content"] = content
            utils.print_ok(f"PASSED - Status: {response.status_code} ({elapsed_time:.2f}s)")
            utils.print_info(f"  Response: {content}")
            utils.print_info(f"  Tokens - Prompt: {usage.get('prompt_tokens', 'N/A')}, Completion: {usage.get('completion_tokens', 'N/A')}")
            # Check auth-type header if present
            auth_type = response.headers.get('UAIG-Auth-Type', response.headers.get('x-auth-type', 'N/A'))
            utils.print_info(f"  Auth-Type: {auth_type}")
        else:
            result["error"] = response.text[:300]
            utils.print_error(f"FAILED - Expected 200, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test1_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test1_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test1_results if r['test_passed'])
utils.print_info(f"\nTest 1 Summary: {passed}/{len(test1_results)} endpoints passed")

<a id='5.3'></a>
### 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)

Send requests with only the API key (no JWT token). Since the access contract has `jwtRequired=true`,
these should be rejected with HTTP 401.

In [ ]:
test2_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 2: API Key Only - No JWT (Expected: 401 Unauthorized)")
utils.print_info(f"{'='*80}")
utils.print_info(f"Using token: {token_source}")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": jwt_api_key,
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=30)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code == 401
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            try:
                error_data = response.json()
                error_msg = error_data.get('error', {}).get('message', 'N/A')
                error_code = error_data.get('error', {}).get('code', 'N/A')
                result["error_code"] = error_code
                utils.print_ok(f"PASSED - Correctly rejected with 401 ({elapsed_time:.2f}s)")
                utils.print_info(f"  Error: {error_msg}")
                utils.print_info(f"  Code: {error_code}")
            except:
                utils.print_ok(f"PASSED - Correctly rejected with 401")
                utils.print_info(f"  Response: {response.text[:200]}")
        else:
            utils.print_error(f"FAILED - Expected 401, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test2_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test2_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test2_results if r['test_passed'])
utils.print_info(f"\nTest 2 Summary: {passed}/{len(test2_results)} endpoints correctly rejected")

<a id='5.4'></a>
### 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)

Send requests with a valid API key but an invalid/expired JWT token.
These should be rejected with HTTP 401.

In [ ]:
test3_results = []
invalid_jwt = "INVALID-TEST-TOKEN-FOR-VALIDATION"

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 3: Invalid JWT Token (Expected: 401 Unauthorized)")
utils.print_info(f"{'='*80}")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": jwt_api_key,
        "Authorization": f"Bearer {invalid_jwt}",
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=30)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code == 401
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            utils.print_ok(f"PASSED - Correctly rejected invalid JWT with 401 ({elapsed_time:.2f}s)")
            utils.print_info(f"  Response: {response.text[:200]}")
        else:
            utils.print_error(f"FAILED - Expected 401, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test3_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test3_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test3_results if r['test_passed'])
utils.print_info(f"\nTest 3 Summary: {passed}/{len(test3_results)} endpoints correctly rejected")

<a id='5.5'></a>
### 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)

Send requests with only a JWT token (no API key). APIM requires a valid subscription key,
so these should be rejected.

In [ ]:
test4_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 4: JWT Only - No API Key (Expected: 401 Unauthorized)")
utils.print_info(f"{'='*80}")
utils.print_info(f"Using token: {token_source}")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "Authorization": f"Bearer {active_token}",
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=30)
        elapsed_time = time.time() - start_time
        
        # APIM rejects without subscription key - could be 401 or 403
        test_passed = response.status_code in [401, 403]
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            utils.print_ok(f"PASSED - Correctly rejected without API key: {response.status_code} ({elapsed_time:.2f}s)")
        else:
            utils.print_error(f"FAILED - Expected 401/403, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test4_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test4_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test4_results if r['test_passed'])
utils.print_info(f"\nTest 4 Summary: {passed}/{len(test4_results)} endpoints correctly rejected")

---
<a id='use-case-2'></a>
## 🛡️ Use Case 2: Role-Enforced Access Contract

> 🔄 [Back to Select Active Token](#3d) — switch between device flow and client credentials before running tests.

This use case creates an access contract that requires **both JWT token and a specific app role**.
The `requiredRoles` variable is set in the product policy, which triggers the `security-handler`
fragment to verify the token's `roles` claim contains at least one of the required roles.

**Prerequisite:** The service principal used in §3 must have the `Models.Read` app role assigned
(via Entra ID portal or CLI group membership).

---

<a id='6.1'></a>
### 6️⃣.1 Define Role-Enforced Access Contract Configuration

In [ ]:
# Role-Enforced Access Contract Configuration
role_contract = {
    "name": f"role-auth-contract-{timestamp}",
    "business_unit": "Security",
    "use_case_name": "RoleAuth",
    "environment": "DEV",
    "use_keyvault": use_keyvault_integration,
    "endpoint_secret": "SEC-ROLE-LLM-ENDPOINT",
    "apikey_secret": "SEC-ROLE-LLM-KEY",
    "description": "Security Role Auth - JWT + App Role enforcement across all endpoints"
}

role_product_id = f"LLM-{role_contract['business_unit']}-{role_contract['use_case_name']}-{role_contract['environment']}"

utils.print_info(f"Role Access Contract Configuration:")
utils.print_info(f"  Business Unit: {role_contract['business_unit']}")
utils.print_info(f"  Use Case: {role_contract['use_case_name']}")
utils.print_info(f"  JWT Enforcement: ENABLED")
utils.print_info(f"  Required Role: Models.Read")
utils.print_info(f"  Product ID: {role_product_id}")

<a id='6.2'></a>
### 6️⃣.2 Create Role-Enforced Product Policy and Deploy

In [ ]:
bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

# Create folder structure for role contract
contract = role_contract
folder_name = f"{contract['business_unit'].lower()}-{contract['use_case_name'].lower()}"
environment_folder = contract['environment'].lower()
role_contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(role_contract_folder, exist_ok=True)
utils.print_info(f"Created folder: {role_contract_folder}")

# Create Role-Enforced Product Policy XML (JWT + requiredRoles)
role_product_policy = '''<policies>
    <inbound>
        <base />
        <!-- Enable JWT requirement for this product -->
        <set-variable name="jwtRequired" value="true" />

        <!-- Enforce app role authorization: only Models.Read role holders can access -->
        <set-variable name="requiredRoles" value="Models.Read" />

        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Setting allowed models variable (comma-separated list) -->
        <set-variable name="allowedModels" value="gpt-4o,gpt-4o-mini,phi-4" />

        <!-- Validate model access based on allowedModels -->
        <include-fragment fragment-id="validate-model-access" />

        <!-- Capacity management -->
        <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="5000" estimate-prompt-tokens="false" token-quota="100000" token-quota-period="Monthly" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

# Write the policy file
role_policy_file = os.path.join(role_contract_folder, "ai-product-policy.xml")
with open(role_policy_file, 'w') as f:
    f.write(role_product_policy)
utils.print_ok(f"Role product policy file created: {role_policy_file}")

# Generate parameter file for role contract
role_params_file = os.path.join(role_contract_folder, "main.bicepparam")
policy_relative_path = "ai-product-policy.xml"

role_params_content = f'''using '../../../main.bicep'

// ============================================================================
// {contract['description']} - Generated from JWT Authentication Testing Notebook
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '{keyvault_subscription_id}'
  resourceGroupName: '{keyvault_resource_group}'
  name: '{keyvault_name}'
}}

param useTargetAzureKeyVault = {str(contract['use_keyvault']).lower()}

param useCase = {{
  businessUnit: '{contract['business_unit']}'
  useCaseName: '{contract['use_case_name']}'
  environment: '{contract['environment']}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: '{contract['endpoint_secret']}'
    apiKeySecretName: '{contract['apikey_secret']}'
    policyXml: loadTextContent('{policy_relative_path}')
  }}
]

param productTerms = 'Role-Enforced Access Contract - {contract["description"]}'

// Azure AI Foundry Integration (disabled)
param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
'''

with open(role_params_file, 'w') as f:
    f.write(role_params_content)
utils.print_ok(f"Parameter file created: {role_params_file}")

# Deploy the role access contract
utils.print_info(f"\n{'='*60}")
utils.print_info(f"Deploying Role-Enforced Access Contract...")
utils.print_info(f"{'='*60}")

deployment_cmd = f"az deployment sub create --name {contract['name']} --location {location} --template-file {template_file} --parameters {role_params_file}"

role_deployment_output = utils.run(
    deployment_cmd,
    f"Deployment '{contract['name']}' succeeded",
    f"Deployment '{contract['name']}' failed"
)

if role_deployment_output.success:
    utils.print_ok(f"Role-Enforced Access Contract deployed successfully!")
else:
    utils.print_error(f"Role-Enforced Access Contract deployment failed!")

<a id='6.3'></a>
### 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract

In [ ]:
# Re-initialize APIM client to pick up new subscriptions
apimClientTool.initialize()

role_subscription_name = f"{role_product_id}-SUB-01"
role_api_key = None

for sub in apimClientTool.apim_subscriptions:
    if role_subscription_name.lower() in sub.get('name', '').lower():
        role_api_key = sub.get('key')
        utils.print_ok(f"Found API key for {role_product_id}")
        break

if not role_api_key:
    utils.print_error(f"Could not find API key for {role_product_id}")

<a id='7.1'></a>
### 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅)

Send requests with both API key and JWT token that contains the required `Models.Read` role.
The `security-handler` should validate the role and allow the request.

In [ ]:
test5_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 5: API Key + JWT with Correct Role (Expected: 200 OK)")
utils.print_info(f"{'='*80}")

utils.print_info(f"Using token: {token_source}")

# Verify the token has the required role before testing
if active_token:
    parts = active_token.split('.')
    if len(parts) >= 2:
        payload = parts[1] + '=' * (4 - len(parts[1]) % 4)
        decoded = json.loads(base64.b64decode(payload))
        token_roles = decoded.get('roles', [])
        if 'Models.Read' in token_roles:
            utils.print_ok(f"Token has 'Models.Read' role - proceeding with test")
        else:
            utils.print_info(f"Token roles: {token_roles or 'none'}")
            utils.print_info(f"WARNING: Token may not have 'Models.Read' role. Test 5 may fail with 403.")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": role_api_key,
        "Authorization": f"Bearer {active_token}",
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=60)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code == 200
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            data = response.json()
            content = data.get('choices', [{}])[0].get('message', {}).get('content', 'N/A')
            utils.print_ok(f"PASSED - Status: {response.status_code} ({elapsed_time:.2f}s)")
            utils.print_info(f"  Response: {content}")
        else:
            result["error"] = response.text[:300]
            utils.print_error(f"FAILED - Expected 200, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test5_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test5_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)


passed = sum(1 for r in test5_results if r['test_passed'])
utils.print_info(f"\nTest 5 Summary: {passed}/{len(test5_results)} endpoints passed")

<a id='7.2'></a>
### 7️⃣.2 Test 6: API Key + JWT Without Required Role (Should Fail ❌)

The `security-handler` should reject the request with HTTP 403 (`insufficient_role`) when the
active token lacks the `Models.Read` role.

> **Note:** If the active token already has `Models.Read` assigned, this test uses an invalid
> token to simulate a missing-role scenario (resulting in 401 instead of 403).

In [ ]:
test6_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 6: API Key + JWT Without Required Role (Expected: 403 Forbidden)")
utils.print_info(f"{'='*80}")
utils.print_info(f"Using token: {token_source}")
# Create a minimal valid-looking JWT without roles to test the role enforcement.
# If the real token already lacks Models.Read, we use it directly.
# Otherwise, we forge a structurally valid but unsigned token to test the 401/403 path.
no_role_token = active_token  # Default: use the active token
use_real_token_without_role = False

if active_token:
    parts = active_token.split('.')
    if len(parts) >= 2:
        pl = parts[1] + '=' * (4 - len(parts[1]) % 4)
        decoded_claims = json.loads(base64.b64decode(pl))
        if 'Models.Read' not in decoded_claims.get('roles', []):
            use_real_token_without_role = True
            utils.print_info("Token does NOT have 'Models.Read' - using it for negative test")

if not use_real_token_without_role:
    utils.print_info("Token already has 'Models.Read' role.")
    utils.print_info("Using an invalid token to simulate missing role (will get 401 instead of 403)")
    # Use an invalid JWT that will fail validation before role check
    no_role_token = "NO-ROLE-TEST-TOKEN-FOR-VALIDATION"

expected_codes = [403] if use_real_token_without_role else [401]

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": role_api_key,
        "Authorization": f"Bearer {no_role_token}",
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=30)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code in expected_codes
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            try:
                error_data = response.json()
                error_msg = error_data.get('error', {}).get('message', 'N/A')
                error_code = error_data.get('error', {}).get('code', 'N/A')
                result["error_code"] = error_code
                utils.print_ok(f"PASSED - Correctly rejected with {response.status_code} ({elapsed_time:.2f}s)")
                utils.print_info(f"  Error: {error_msg}")
                utils.print_info(f"  Code: {error_code}")
                if error_code == "insufficient_role":
                    utils.print_info(f"  Required: {error_data.get('error', {}).get('required_roles', 'N/A')}")
                    utils.print_info(f"  Token roles: {error_data.get('error', {}).get('token_roles', 'N/A')}")
            except:
                utils.print_ok(f"PASSED - Correctly rejected with {response.status_code}")
        else:
            utils.print_error(f"FAILED - Expected {expected_codes}, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test6_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test6_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test6_results if r['test_passed'])
utils.print_info(f"\nTest 6 Summary: {passed}/{len(test6_results)} endpoints correctly rejected")

<a id='7.3'></a>
### 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)

Send requests with only the API key (no JWT). Since the product has `jwtRequired=true`,
the request should be rejected with 401 before role checking even occurs.

In [ ]:
test7_results = []

utils.print_info(f"\n{'='*80}")
utils.print_info(f"TEST 7: API Key Only on Role-Enforced Product (Expected: 401 Unauthorized)")
utils.print_info(f"{'='*80}")

for i, ep in enumerate(api_endpoints, 1):
    utils.print_info(f"\n--- Endpoint {i}/{len(api_endpoints)}: {ep['name']} ---")
    
    headers = {
        "api-key": role_api_key,
        "Content-Type": "application/json"
    }
    
    try:
        start_time = time.time()
        response = requests.post(ep['url'], headers=headers, json=ep['payload'], timeout=30)
        elapsed_time = time.time() - start_time
        
        test_passed = response.status_code == 401
        result = {
            "endpoint": ep['name'],
            "status_code": response.status_code,
            "elapsed_time": elapsed_time,
            "test_passed": test_passed
        }
        
        if test_passed:
            try:
                error_data = response.json()
                error_code = error_data.get('error', {}).get('code', 'N/A')
                utils.print_ok(f"PASSED - Correctly rejected with 401 ({elapsed_time:.2f}s)")
                utils.print_info(f"  Code: {error_code}")
            except:
                utils.print_ok(f"PASSED - Correctly rejected with 401")
        else:
            utils.print_error(f"FAILED - Expected 401, got {response.status_code}")
            utils.print_error(f"  Response: {response.text[:300]}")
        
        test7_results.append(result)
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
        test7_results.append({"endpoint": ep['name'], "status_code": 0, "test_passed": False, "error": str(e)})
    
    time.sleep(1)

passed = sum(1 for r in test7_results if r['test_passed'])
utils.print_info(f"\nTest 7 Summary: {passed}/{len(test7_results)} endpoints correctly rejected")

---
## 📊 Results Summary
---

In [ ]:
utils.print_info(f"\n{'='*80}")
utils.print_info(f"OVERALL JWT AUTHENTICATION & ROLE AUTHORIZATION TEST SUMMARY")
utils.print_info(f"{'='*80}")

all_results = {
    "Test 1: API Key + JWT (200)": test1_results,
    "Test 2: API Key Only (401)": test2_results,
    "Test 3: Invalid JWT (401)": test3_results,
    "Test 4: JWT Only (401/403)": test4_results,
}

# Include role-based tests if they were executed
try:
    if test5_results:
        all_results["Test 5: Role JWT + Correct Role (200)"] = test5_results
except NameError:
    pass
try:
    if test6_results:
        all_results["Test 6: Role JWT + Wrong/No Role (403)"] = test6_results
except NameError:
    pass
try:
    if test7_results:
        all_results["Test 7: Role Product + No JWT (401)"] = test7_results
except NameError:
    pass

total_tests = 0
total_passed = 0

for test_name, results in all_results.items():
    passed = sum(1 for r in results if r['test_passed'])
    total = len(results)
    total_tests += total
    total_passed += passed
    
    status = "PASS" if passed == total else "FAIL"
    utils.print_info(f"\n[{status}] {test_name}:")
    for r in results:
        ep_status = "PASS" if r['test_passed'] else "FAIL"
        utils.print_info(f"   [{ep_status}] {r['endpoint']}: HTTP {r['status_code']}")

utils.print_info(f"\n{'='*80}")
utils.print_info(f"Overall Results:")
utils.print_info(f"   Total tests executed: {total_tests}")
if total_passed == total_tests:
    utils.print_ok(f"   All tests passed! ({total_passed}/{total_tests})")
else:
    utils.print_info(f"   Passed: {total_passed}/{total_tests}")
    utils.print_error(f"   Failed: {total_tests - total_passed}/{total_tests}")

utils.print_info(f"\nUse Case 1 Product ID: {jwt_product_id}")
try:
    utils.print_info(f"Use Case 2 Product ID: {role_product_id}")
except NameError:
    pass
utils.print_info(f"Endpoints tested: {len(api_endpoints)} (Azure OpenAI, Universal LLM, Unified AI)")

<a id='cleanup'></a>
### Cleanup (Optional)

Remove the test JWT access contract from APIM created during this notebook session.

> **Note:** This will not delete any Entra ID app registrations or APIM named values.

In [ ]:
# Set to True to delete the access contracts created in this session
cleanup_enabled = True

if cleanup_enabled:
    # Clean up JWT product (Use Case 1)
    utils.print_info(f"Deleting JWT product: {jwt_product_id}...")
    prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {jwt_product_id} --delete-subscriptions true --yes"
    utils.run(prod_cmd, f"Deleted product {jwt_product_id}", f"Failed to delete product {jwt_product_id}")

    if os.path.exists(jwt_contract_folder):
        shutil.rmtree(jwt_contract_folder)
        utils.print_info(f"Removed folder: {jwt_contract_folder}")

    # Clean up Role product (Use Case 2)
    try:
        utils.print_info(f"Deleting Role product: {role_product_id}...")
        prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {role_product_id} --delete-subscriptions true --yes"
        utils.run(prod_cmd, f"Deleted product {role_product_id}", f"Failed to delete product {role_product_id}")

        if os.path.exists(role_contract_folder):
            shutil.rmtree(role_contract_folder)
            utils.print_info(f"Removed folder: {role_contract_folder}")
    except NameError:
        utils.print_info("Role product was not deployed in this session - skipping")

    utils.print_ok("Cleanup completed!")
else:
    utils.print_info("Cleanup is disabled. Set cleanup_enabled = True to remove test resources.")